In [ ]:
import os
import requests
import re
import sys 


# In[94]:


# Code here - Import BeautifulSoup library
from bs4 import BeautifulSoup
# Code ends here


# In[95]:


# function to get the html source text of the medium article
def get_page():
    global url
    url = input("Enter url of a medium article: ")
    if not (re.match(r'https?://medium.com/',url)
        or re.match(r'https?://web.archive.org/web/.*/https?://medium.com/', url)):
        print('Please enter a valid website, or make sure it is a medium article')
        sys.exit(1)
    res = requests.get(url)
    print(res.text)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, 'html.parser')
    return soup


# In[96]:


def clean(text):
    rep = {"<br>": "\n", "<br/>": "\n", "<li>":  "\n"}
    rep = dict((re.escape(k), v) for k, v in rep.items()) 
    pattern = re.compile("|".join(rep.keys()))
    text = pattern.sub(lambda m: rep[re.escape(m.group(0))], text)
    text = re.sub('\<(.*?)\>', '', text)
    return text



def collect_text(soup):
    text = f'url: {url}\n\n'
    para_text = soup.find_all('p')
    #print(f"paragraphs text = \n {para_text}")
    for para in para_text:
        if para.get('id'):
            clean_para = para.get_text(strip=True)
            if clean_para:
                text += f"{para.text}\n\n"
    return text


# In[97]:


# function to save file in the current directory
def save_file(text):
    if not os.path.exists('./scraped_articles'):
        os.mkdir('./scraped_articles')
    name = url.split("/")[-1]
    print(name)
    fname = f'scraped_articles/{name}.txt'
    with open (fname,'w') as fw:
        fw.write(text)
        print(f'File saved in directory {fname}')


# In[98]:


if __name__ == '__main__':
	text = collect_text(get_page())
	save_file(text)
	# Instructions to Run this python code
	# Give url as https://medium.com/@subashgandyer/papa-what-is-a-neural-network-c5e5cc427c7